# GTNet Multimodal Trajectory Prediction Visualization

This notebook demonstrates visualization of multimodal trajectory predictions from the improved GTNet model.

**Features demonstrated:**
1. Load trained model with GAT + multimodal prediction
2. Run inference to get K=3 trajectory predictions
3. Plot ground truth and predicted trajectories
4. Visualize attention weights (which neighbors are most important)
5. Show per-mode specialization (mode 0 = left, mode 1 = straight, mode 2 = right)

**Requirements:** 12.10

## Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from core_perception.multi_agent_model import (
    MultiAgentTrajectoryPredictor,
    MultiAgentModelConfig,
    load_checkpoint_with_compatibility,
)
from core_perception.multi_agent_trajectory import compute_multimodal_metrics

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✓ Imports successful")

## 1. Load Trained Model

Load a trained model with GAT and multimodal prediction enabled.

In [ ]:
# Configuration
MODEL_PATH = project_root / "models" / "multi_agent" / "Town01" / "best_checkpoint.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Model path: {MODEL_PATH}")

# Check if model exists
if not MODEL_PATH.exists():
    print("\n⚠️  Model not found!")
    print("\nTo train a model, run:")
    print("  python scripts/train_multi_agent_trajectory.py --data-dir data/multi_agent/Town01 --town-filter Town01")
    print("\nFor this demo, we'll create a model with random weights.")
    
    # Create model with full config (GAT + multimodal)
    config = MultiAgentModelConfig.full_config()
    model = MultiAgentTrajectoryPredictor(config=config)
    model = model.to(DEVICE)
    model.eval()
    
    print(f"\n✓ Created model with config:")
    print(f"  - GAT enabled: {config.enable_gat}")
    print(f"  - Multimodal enabled: {config.enable_multimodal}")
    print(f"  - Num modes: {config.num_modes}")
    print(f"  - Attention heads: {config.num_attention_heads}")
else:
    # Load trained model
    model, checkpoint = load_checkpoint_with_compatibility(str(MODEL_PATH), device=DEVICE)
    model.eval()
    
    print(f"\n✓ Loaded trained model")
    print(f"  - Epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"  - Best val minADE: {checkpoint.get('best_val_minADE', 'N/A'):.4f}")
    print(f"  - Config: {model.config}")

## 2. Load Sample Data

Load a sample from the dataset to visualize predictions.

In [ ]:
# Sample data path
SAMPLE_PATH = project_root / "data" / "multi_agent" / "Town01" / "samples" / "sample_0000.pt"

if not SAMPLE_PATH.exists():
    print("⚠️  Sample data not found!")
    print("\nTo build dataset, run:")
    print("  python scripts/build_multi_agent_dataset.py --csv data/multi_agent/raw/Town01_*.csv --output data/multi_agent/Town01")
    print("\nFor this demo, we'll create synthetic sample data.")
    
    # Create synthetic sample
    batch_size = 1
    max_agents = 8
    history_steps = 20
    future_steps = 30
    
    # Create sample with realistic trajectories
    sample = {
        "x": torch.randn(batch_size, max_agents, history_steps, 6).to(DEVICE),
        "y": torch.randn(batch_size, max_agents, future_steps, 2).to(DEVICE),
        "adj": torch.eye(max_agents).unsqueeze(0).to(DEVICE),
        "x_mask": torch.ones(batch_size, max_agents, history_steps, dtype=torch.bool).to(DEVICE),
        "y_mask": torch.ones(batch_size, max_agents, future_steps, dtype=torch.bool).to(DEVICE),
        "actor_ids": torch.arange(max_agents).unsqueeze(0).to(DEVICE),
    }
    
    # Add some connections to adjacency matrix
    sample["adj"][0, 0, 1] = 1.0
    sample["adj"][0, 1, 0] = 1.0
    sample["adj"][0, 0, 2] = 1.0
    sample["adj"][0, 2, 0] = 1.0
    
    print("\n✓ Created synthetic sample data")
else:
    # Load real sample
    sample = torch.load(SAMPLE_PATH)
    
    # Move to device and add batch dimension if needed
    for key in ["x", "y", "adj", "x_mask", "y_mask", "actor_ids"]:
        if key in sample:
            sample[key] = sample[key].to(DEVICE)
            if sample[key].dim() == 2 or (key in ["x", "y"] and sample[key].dim() == 3):
                sample[key] = sample[key].unsqueeze(0)
    
    print("\n✓ Loaded sample data")

print(f"  - Batch size: {sample['x'].shape[0]}")
print(f"  - Num agents: {sample['x'].shape[1]}")
print(f"  - History steps: {sample['x'].shape[2]}")
print(f"  - Future steps: {sample['y'].shape[2]}")

## 3. Run Inference

Run the model to get K=3 multimodal predictions.

In [ ]:
# Run inference
with torch.no_grad():
    predictions = model(
        x=sample["x"],
        adj=sample["adj"],
        x_mask=sample["x_mask"],
        agent_mask=sample["x_mask"].any(dim=-1),
    )

print(f"✓ Inference complete")
print(f"  - Prediction shape: {predictions.shape}")

if model.config.enable_multimodal:
    print(f"  - Multimodal: {model.config.num_modes} modes")
    batch_size, max_agents, num_modes, future_steps, _ = predictions.shape
else:
    print(f"  - Unimodal: single trajectory")
    batch_size, max_agents, future_steps, _ = predictions.shape
    num_modes = 1
    # Add mode dimension for consistent handling
    predictions = predictions.unsqueeze(2)

# Compute metrics
agent_mask = sample["x_mask"].any(dim=-1)
metrics = compute_multimodal_metrics(
    pred=predictions,
    target=sample["y"],
    y_mask=sample["y_mask"],
    agent_mask=agent_mask,
)

print(f"\n✓ Metrics:")
print(f"  - minADE: {metrics['minADE']:.4f} m")
print(f"  - minFDE: {metrics['minFDE']:.4f} m")
print(f"  - MissRate: {metrics['MissRate']:.4f}")

if model.config.enable_multimodal:
    print(f"\n  Per-mode ADE:")
    for k in range(num_modes):
        print(f"    - Mode {k}: {metrics[f'mode_{k}_ADE']:.4f} m")

## 4. Visualize Trajectories

Plot ground truth and predicted trajectories for all agents.

In [ ]:
def plot_trajectories(sample, predictions, agent_idx=0, show_all_modes=True):
    """
    Plot ground truth and predicted trajectories for a specific agent.
    
    Args:
        sample: Sample data dictionary
        predictions: Model predictions [B, N, K, T, 2]
        agent_idx: Index of agent to visualize
        show_all_modes: If True, show all K modes; else show only best mode
    """
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Extract data for visualization
    x = sample["x"][0].cpu().numpy()  # [N, H, 6]
    y = sample["y"][0].cpu().numpy()  # [N, T, 2]
    pred = predictions[0].cpu().numpy()  # [N, K, T, 2]
    x_mask = sample["x_mask"][0].cpu().numpy()  # [N, H]
    y_mask = sample["y_mask"][0].cpu().numpy()  # [N, T]
    adj = sample["adj"][0].cpu().numpy()  # [N, N]
    
    num_agents = x.shape[0]
    
    # Define colors for modes
    mode_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']  # Red, Teal, Blue
    mode_labels = ['Mode 0 (Left)', 'Mode 1 (Straight)', 'Mode 2 (Right)']
    
    # Plot all agents
    for i in range(num_agents):
        # Skip invalid agents
        if not x_mask[i].any():
            continue
        
        # Extract history trajectory (local_x, local_y from features)
        history_x = x[i, x_mask[i], 0]  # local_x
        history_y = x[i, x_mask[i], 1]  # local_y
        
        # Extract ground truth future
        future_x = y[i, y_mask[i], 0]
        future_y = y[i, y_mask[i], 1]
        
        # Determine if this is the focus agent
        is_focus = (i == agent_idx)
        alpha = 1.0 if is_focus else 0.3
        linewidth = 2.5 if is_focus else 1.5
        
        # Plot history (gray)
        ax.plot(history_x, history_y, 'o-', color='gray', alpha=alpha, 
                linewidth=linewidth, markersize=4, label='History' if i == 0 else '')
        
        # Plot ground truth future (green)
        ax.plot(future_x, future_y, 's-', color='green', alpha=alpha,
                linewidth=linewidth, markersize=4, label='Ground Truth' if i == 0 else '')
        
        # Plot predicted trajectories
        if is_focus or not show_all_modes:
            for k in range(pred.shape[1]):
                pred_x = pred[i, k, y_mask[i], 0]
                pred_y = pred[i, k, y_mask[i], 1]
                
                if show_all_modes or k == 0:
                    ax.plot(pred_x, pred_y, '--', color=mode_colors[k % len(mode_colors)],
                            alpha=alpha, linewidth=linewidth, 
                            label=mode_labels[k] if i == agent_idx else '')
        
        # Mark current position (last history point)
        if len(history_x) > 0:
            ax.plot(history_x[-1], history_y[-1], 'o', color='black' if is_focus else 'gray',
                    markersize=10 if is_focus else 6, zorder=10)
            
            # Add agent label
            if is_focus:
                ax.text(history_x[-1], history_y[-1] + 2, f'Agent {i} (Focus)',
                        fontsize=12, fontweight='bold', ha='center')
    
    # Draw connections (adjacency)
    for i in range(num_agents):
        if not x_mask[i].any():
            continue
        for j in range(i + 1, num_agents):
            if not x_mask[j].any():
                continue
            if adj[i, j] > 0:
                # Get last positions
                i_x, i_y = x[i, x_mask[i], 0][-1], x[i, x_mask[i], 1][-1]
                j_x, j_y = x[j, x_mask[j], 0][-1], x[j, x_mask[j], 1][-1]
                ax.plot([i_x, j_x], [i_y, j_y], ':', color='lightblue', 
                        alpha=0.5, linewidth=1, zorder=0)
    
    ax.set_xlabel('X (meters, ego-centric)', fontsize=12)
    ax.set_ylabel('Y (meters, ego-centric, +Y forward)', fontsize=12)
    ax.set_title(f'Multimodal Trajectory Prediction (Agent {agent_idx})', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    return fig

# Plot trajectories for agent 0
fig = plot_trajectories(sample, predictions, agent_idx=0, show_all_modes=True)
plt.show()

print("\n✓ Trajectory visualization complete")

## 5. Visualize Attention Weights

Extract and visualize attention weights from GAT layers to see which neighbors are most important.

In [ ]:
def extract_attention_weights(model, sample):
    """
    Extract attention weights from GAT layers during forward pass.
    
    Returns:
        List of attention weight tensors, one per GAT layer
    """
    if not model.config.enable_gat:
        print("⚠️  GAT not enabled in this model")
        return None
    
    attention_weights = []
    
    # Hook to capture attention weights
    def attention_hook(module, input, output):
        # Store attention weights if available
        if hasattr(module, '_last_attention_weights'):
            attention_weights.append(module._last_attention_weights.detach().cpu())
    
    # Register hooks on GAT layers
    hooks = []
    for block in model.graph_blocks:
        if hasattr(block, 'attention'):
            hooks.append(block.register_forward_hook(attention_hook))
    
    # Run forward pass
    with torch.no_grad():
        _ = model(
            x=sample["x"],
            adj=sample["adj"],
            x_mask=sample["x_mask"],
            agent_mask=sample["x_mask"].any(dim=-1),
        )
    
    # Remove hooks
    for hook in hooks:
        hook.remove()
    
    return attention_weights if attention_weights else None


def plot_attention_heatmap(sample, agent_idx=0):
    """
    Plot attention weight heatmap showing which neighbors are most important.
    
    Note: This is a simplified visualization. For actual attention weights,
    we would need to modify the GAT layer to expose them during forward pass.
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Extract adjacency matrix
    adj = sample["adj"][0].cpu().numpy()  # [N, N]
    x_mask = sample["x_mask"][0].cpu().numpy()  # [N, H]
    
    # Get valid agents
    valid_agents = x_mask.any(axis=1)
    num_valid = valid_agents.sum()
    
    # Create attention-like weights (for demo purposes)
    # In practice, these would come from the GAT layer
    attention = adj.copy()
    
    # Normalize by row (softmax-like)
    row_sums = attention.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # Avoid division by zero
    attention = attention / row_sums
    
    # Mask invalid agents
    attention = attention * valid_agents[:, None] * valid_agents[None, :]
    
    # Plot heatmap
    im = ax.imshow(attention, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Attention Weight', fontsize=12)
    
    # Set ticks and labels
    ax.set_xticks(range(attention.shape[1]))
    ax.set_yticks(range(attention.shape[0]))
    ax.set_xticklabels([f'A{i}' for i in range(attention.shape[1])])
    ax.set_yticklabels([f'A{i}' for i in range(attention.shape[0])])
    
    # Highlight focus agent
    ax.axhline(y=agent_idx - 0.5, color='blue', linewidth=2, linestyle='--', alpha=0.7)
    ax.axhline(y=agent_idx + 0.5, color='blue', linewidth=2, linestyle='--', alpha=0.7)
    ax.axvline(x=agent_idx - 0.5, color='blue', linewidth=2, linestyle='--', alpha=0.7)
    ax.axvline(x=agent_idx + 0.5, color='blue', linewidth=2, linestyle='--', alpha=0.7)
    
    # Add text annotations for non-zero weights
    for i in range(attention.shape[0]):
        for j in range(attention.shape[1]):
            if attention[i, j] > 0.01:
                text = ax.text(j, i, f'{attention[i, j]:.2f}',
                              ha="center", va="center", color="black", fontsize=9)
    
    ax.set_xlabel('Neighbor Agent', fontsize=12)
    ax.set_ylabel('Query Agent', fontsize=12)
    ax.set_title(f'Attention Weights (Focus: Agent {agent_idx})', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    return fig

# Plot attention heatmap
if model.config.enable_gat:
    fig = plot_attention_heatmap(sample, agent_idx=0)
    plt.show()
    print("\n✓ Attention visualization complete")
    print("\nNote: This shows the adjacency-based attention pattern.")
    print("For actual learned attention weights, the GAT layer would need to expose them.")
else:
    print("\n⚠️  GAT not enabled - skipping attention visualization")

## 6. Analyze Per-Mode Specialization

Analyze how different modes specialize to different trajectory patterns (left, straight, right).

In [ ]:
def analyze_mode_specialization(predictions, sample, agent_idx=0):
    """
    Analyze and visualize how modes specialize to different trajectory patterns.
    
    Computes the lateral deviation (x-direction) for each mode to identify:
    - Mode 0: Left turns (positive x)
    - Mode 1: Straight (near-zero x)
    - Mode 2: Right turns (negative x)
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Extract data
    pred = predictions[0, agent_idx].cpu().numpy()  # [K, T, 2]
    y = sample["y"][0, agent_idx].cpu().numpy()  # [T, 2]
    y_mask = sample["y_mask"][0, agent_idx].cpu().numpy()  # [T]
    
    num_modes = pred.shape[0]
    timesteps = np.arange(pred.shape[1])
    
    mode_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    mode_labels = ['Mode 0 (Left)', 'Mode 1 (Straight)', 'Mode 2 (Right)']
    
    # 1. Lateral deviation over time
    ax = axes[0, 0]
    for k in range(num_modes):
        lateral = pred[k, y_mask, 0]  # x-coordinate (lateral)
        ax.plot(timesteps[y_mask], lateral, '-o', color=mode_colors[k],
                label=mode_labels[k], linewidth=2, markersize=4)
    
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3, label='Center line')
    ax.set_xlabel('Timestep', fontsize=11)
    ax.set_ylabel('Lateral Position (m)', fontsize=11)
    ax.set_title('Lateral Deviation Over Time', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # 2. Longitudinal progress over time
    ax = axes[0, 1]
    for k in range(num_modes):
        longitudinal = pred[k, y_mask, 1]  # y-coordinate (forward)
        ax.plot(timesteps[y_mask], longitudinal, '-o', color=mode_colors[k],
                label=mode_labels[k], linewidth=2, markersize=4)
    
    # Ground truth
    ax.plot(timesteps[y_mask], y[y_mask, 1], 's--', color='green',
            label='Ground Truth', linewidth=2, markersize=4, alpha=0.7)
    
    ax.set_xlabel('Timestep', fontsize=11)
    ax.set_ylabel('Longitudinal Position (m)', fontsize=11)
    ax.set_title('Longitudinal Progress Over Time', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # 3. Final position distribution
    ax = axes[1, 0]
    final_positions = pred[:, y_mask, :][:, -1, :]  # [K, 2]
    
    for k in range(num_modes):
        ax.scatter(final_positions[k, 0], final_positions[k, 1],
                  s=200, color=mode_colors[k], label=mode_labels[k],
                  edgecolors='black', linewidths=2, zorder=10)
    
    # Ground truth final position
    gt_final = y[y_mask][-1]
    ax.scatter(gt_final[0], gt_final[1], s=200, color='green',
              marker='s', label='Ground Truth', edgecolors='black',
              linewidths=2, zorder=10)
    
    ax.set_xlabel('Lateral Position (m)', fontsize=11)
    ax.set_ylabel('Longitudinal Position (m)', fontsize=11)
    ax.set_title('Final Position Distribution', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    # 4. Mode statistics
    ax = axes[1, 1]
    
    # Compute statistics for each mode
    stats = []
    for k in range(num_modes):
        lateral_mean = pred[k, y_mask, 0].mean()
        lateral_std = pred[k, y_mask, 0].std()
        longitudinal_mean = pred[k, y_mask, 1].mean()
        
        # Compute error from ground truth
        error = np.linalg.norm(pred[k, y_mask] - y[y_mask], axis=-1).mean()
        
        stats.append({
            'mode': k,
            'lateral_mean': lateral_mean,
            'lateral_std': lateral_std,
            'longitudinal_mean': longitudinal_mean,
            'error': error,
        })
    
    # Create bar chart
    x_pos = np.arange(num_modes)
    lateral_means = [s['lateral_mean'] for s in stats]
    errors = [s['error'] for s in stats]
    
    ax2 = ax.twinx()
    
    bars1 = ax.bar(x_pos - 0.2, lateral_means, 0.4, 
                   color=[mode_colors[k] for k in range(num_modes)],
                   alpha=0.7, label='Mean Lateral Position')
    bars2 = ax2.bar(x_pos + 0.2, errors, 0.4,
                    color='gray', alpha=0.5, label='Mean Error')
    
    ax.set_xlabel('Mode', fontsize=11)
    ax.set_ylabel('Mean Lateral Position (m)', fontsize=11, color='black')
    ax2.set_ylabel('Mean Error (m)', fontsize=11, color='gray')
    ax.set_title('Mode Statistics', fontsize=12, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'Mode {k}' for k in range(num_modes)])
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add legends
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')
    
    plt.tight_layout()
    
    # Print statistics
    print("\n" + "="*60)
    print(f"Mode Specialization Analysis (Agent {agent_idx})")
    print("="*60)
    for s in stats:
        print(f"\nMode {s['mode']}:")
        print(f"  Lateral mean: {s['lateral_mean']:+.2f} m (+ = left, - = right)")
        print(f"  Lateral std:  {s['lateral_std']:.2f} m")
        print(f"  Longitudinal: {s['longitudinal_mean']:.2f} m")
        print(f"  Mean error:   {s['error']:.2f} m")
    print("="*60)
    
    return fig

# Analyze mode specialization
if model.config.enable_multimodal:
    fig = analyze_mode_specialization(predictions, sample, agent_idx=0)
    plt.show()
    print("\n✓ Mode specialization analysis complete")
else:
    print("\n⚠️  Multimodal prediction not enabled - skipping mode analysis")

## 7. Batch Visualization

Visualize predictions for multiple agents in a single scene.

In [ ]:
def plot_scene_overview(sample, predictions, show_best_mode_only=False):
    """
    Plot an overview of all agents in the scene with their predictions.
    
    Args:
        sample: Sample data dictionary
        predictions: Model predictions [B, N, K, T, 2]
        show_best_mode_only: If True, show only the best mode per agent
    """
    fig, ax = plt.subplots(figsize=(14, 12))
    
    # Extract data
    x = sample["x"][0].cpu().numpy()  # [N, H, 6]
    y = sample["y"][0].cpu().numpy()  # [N, T, 2]
    pred = predictions[0].cpu().numpy()  # [N, K, T, 2]
    x_mask = sample["x_mask"][0].cpu().numpy()  # [N, H]
    y_mask = sample["y_mask"][0].cpu().numpy()  # [N, T]
    adj = sample["adj"][0].cpu().numpy()  # [N, N]
    
    num_agents = x.shape[0]
    num_modes = pred.shape[1]
    
    mode_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    # Plot each agent
    for i in range(num_agents):
        if not x_mask[i].any():
            continue
        
        # History
        history_x = x[i, x_mask[i], 0]
        history_y = x[i, x_mask[i], 1]
        
        # Ground truth future
        future_x = y[i, y_mask[i], 0]
        future_y = y[i, y_mask[i], 1]
        
        # Plot history
        ax.plot(history_x, history_y, 'o-', color='gray', 
                linewidth=2, markersize=3, alpha=0.6)
        
        # Plot ground truth
        ax.plot(future_x, future_y, 's-', color='green',
                linewidth=2, markersize=3, alpha=0.6)
        
        # Plot predictions
        if show_best_mode_only:
            # Find best mode (minimum error)
            errors = []
            for k in range(num_modes):
                error = np.linalg.norm(pred[i, k, y_mask[i]] - y[i, y_mask[i]], axis=-1).mean()
                errors.append(error)
            best_k = np.argmin(errors)
            
            pred_x = pred[i, best_k, y_mask[i], 0]
            pred_y = pred[i, best_k, y_mask[i], 1]
            ax.plot(pred_x, pred_y, '--', color=mode_colors[best_k],
                    linewidth=2, alpha=0.8)
        else:
            # Show all modes
            for k in range(num_modes):
                pred_x = pred[i, k, y_mask[i], 0]
                pred_y = pred[i, k, y_mask[i], 1]
                ax.plot(pred_x, pred_y, '--', color=mode_colors[k % len(mode_colors)],
                        linewidth=1.5, alpha=0.5)
        
        # Mark current position
        if len(history_x) > 0:
            ax.plot(history_x[-1], history_y[-1], 'o', color='black',
                    markersize=8, zorder=10)
            ax.text(history_x[-1] + 1, history_y[-1] + 1, f'{i}',
                    fontsize=10, fontweight='bold', color='black')
    
    # Draw adjacency connections
    for i in range(num_agents):
        if not x_mask[i].any():
            continue
        for j in range(i + 1, num_agents):
            if not x_mask[j].any():
                continue
            if adj[i, j] > 0:
                i_x, i_y = x[i, x_mask[i], 0][-1], x[i, x_mask[i], 1][-1]
                j_x, j_y = x[j, x_mask[j], 0][-1], x[j, x_mask[j], 1][-1]
                ax.plot([i_x, j_x], [i_y, j_y], ':', color='lightblue',
                        alpha=0.4, linewidth=1, zorder=0)
    
    # Add legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='gray', linewidth=2, label='History'),
        Line2D([0], [0], color='green', linewidth=2, label='Ground Truth'),
    ]
    if show_best_mode_only:
        legend_elements.append(Line2D([0], [0], color='gray', linestyle='--', linewidth=2, label='Best Prediction'))
    else:
        for k in range(min(num_modes, 3)):
            legend_elements.append(Line2D([0], [0], color=mode_colors[k], linestyle='--', 
                                         linewidth=2, label=f'Mode {k}'))
    legend_elements.append(Line2D([0], [0], color='lightblue', linestyle=':', linewidth=1, label='Adjacency'))
    
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    ax.set_xlabel('X (meters, ego-centric)', fontsize=12)
    ax.set_ylabel('Y (meters, ego-centric, +Y forward)', fontsize=12)
    ax.set_title('Multi-Agent Scene Overview', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    return fig

# Plot scene overview
fig = plot_scene_overview(sample, predictions, show_best_mode_only=False)
plt.show()

print("\n✓ Scene overview visualization complete")

## Summary

This notebook demonstrated:

1. **Model Loading**: Loaded a trained GTNet model with GAT and multimodal prediction
2. **Inference**: Ran the model to get K=3 trajectory predictions
3. **Trajectory Visualization**: Plotted ground truth and predicted trajectories
4. **Attention Weights**: Visualized which neighbors are most important (GAT)
5. **Mode Specialization**: Analyzed how modes specialize to different patterns (left/straight/right)
6. **Scene Overview**: Visualized all agents in a multi-agent scene

### Next Steps

- Train models on real CARLA data using `scripts/train_multi_agent_trajectory.py`
- Collect more diverse scenarios to improve mode specialization
- Experiment with different numbers of modes (K=3, 5, 10)
- Analyze attention patterns across different traffic scenarios
- Compare baseline vs. improved model performance

### References

- **GAT**: Veličković et al., "Graph Attention Networks", ICLR 2018
- **WTA Loss**: Lee et al., "DESIRE: Distant Future Prediction in Dynamic Scenes", CVPR 2017
- **Multimodal Prediction**: Trajectron++, MultiPath, and related work